<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2012%20-%202026-05-08%20-%20Correlaciones%2C%20regresiones%20e%20intro%20a%20scikit-learn/clase_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 12 · Correlaciones, regresiones e intro a scikit-learn · 🧪 Taller

**Fecha:** Viernes 8 de mayo de 2026 · **Duración estimada:** ~60 min · **Nivel:** Intermedio

> 📖 **La teoría completa vive en el [HTML de la sesión](./index.html).** Este cuaderno es el **taller práctico**: aquí codificas.

## 🎯 Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

1. **Analizar** correlaciones bivariadas (Pearson, Spearman) e interpretar correctamente.
2. **Construir** modelos de regresión lineal y logística con scikit-learn.
3. **Diseñar** un pipeline train/test correcto (sin data leakage).
4. **Evaluar** modelos con métricas apropiadas (R², accuracy, MAE).
5. **Interpretar** coeficientes y extraer features importantes.

## ✅ Prerrequisitos

- Días 10-11: visualización y tests estadísticos.
- Pandas: `groupby()`, selección de columnas.
- Numpy: arrays básicos.

## 🧭 Cómo usar este cuaderno

- Ejecuta con `Shift + Enter`.
- Ejercicios: 🟢 guiado, 🟡 aplicado, 🔴 desafío.
- Soluciones ocultas: `# %% [solution]`.
- `assert` valida código: sin error = ✓.

---

# Setup

In [ ]:
# --- Setup del entorno ---------------------------------------------------

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")
print(f"scikit-learn {__import__('sklearn').__version__}")

---

## 🎬 Hook · ¿Predecir la tarifa del Titanic con edad y clase?

> **Pregunta de hoy:** Sabemos que edad y clase se relacionan con la tarifa. ¿Podemos **predecir** la tarifa de un pasajero dado su edad, clase y sexo? ¿Cuán bien funciona?

In [ ]:
# Cargar Titanic
titanic = sns.load_dataset("titanic").dropna(subset=["age", "fare"])
print(f"Shape: {titanic.shape}")
print(f"\nPrimeras 3 filas:")
titanic[["age", "pclass", "sex", "fare"]].head(3)

---

## 1. Correlación de Pearson

⏱️ ~8 min

> 📌 **Concepto clave:** Pearson mide asociación **lineal**. Rango [-1, +1]: +1 = relación lineal perfecta positiva, -1 = perfecta negativa, 0 = sin relación lineal.

In [ ]:
# Demo: correlación simple
r_age_fare = titanic["age"].corr(titanic["fare"])
print(f"Pearson(edad, tarifa) = {r_age_fare:.3f}")

# Con p-valor
r, p = stats.pearsonr(titanic["age"], titanic["fare"])
print(f"Pearson r = {r:.3f}, p-value = {p:.4f}")
print(f"Significativa: {'sí' if p < 0.05 else 'no'}")

# Visualizar
plt.figure(figsize=(10, 4))
plt.scatter(titanic["age"], titanic["fare"], alpha=0.5, s=30)
# Agregar línea de regresión
z = np.polyfit(titanic["age"], titanic["fare"], 1)
p_fit = np.poly1d(z)
plt.plot(titanic["age"].sort_values(), p_fit(titanic["age"].sort_values()), "r-", lw=2, label="Fit lineal")
plt.xlabel("Edad (años)")
plt.ylabel("Tarifa (£)")
plt.title(f"Edad vs Tarifa (r={r:.3f})")
plt.legend()
plt.tight_layout()

### 🟢 Ejercicio 1 — Guiado · Matriz de correlación

⏱️ ~5 min

Crea una matriz de correlación de Pearson para las columnas numéricas del Titanic: `age`, `fare`, `pclass`, `survived`. Visualiza con `sns.heatmap()`.

In [ ]:
# 🟢 Correlación matriz

cols_numericas = ["age", "fare", "pclass", "survived"]
corr_matrix = titanic[cols_numericas].___()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación — Titanic")
plt.tight_layout()

In [ ]:
assert corr_matrix.shape == (4, 4)
print("✅ Ejercicio 1 completado")

In [ ]:
# %% [solution]
# corr_matrix = titanic[cols_numericas].corr()

---

## 2. Regresión lineal · modelo base

⏱️ ~12 min

> 📌 **Concepto clave:** Regresión lineal ajusta `y = β₀ + β₁·x` minimizando error cuadrático. Fácil de interpretar: cada coeficiente β \= cambio en y cuando x aumenta 1 unidad.

In [ ]:
# Demo: regresión simple (edad → tarifa)

X = titanic[["age"]].values  # variable independiente (2D)
y = titanic["fare"].values   # variable dependiente (1D)

# Entrenar
modelo = LinearRegression()
modelo.fit(X, y)

# Parámetros
intercept = modelo.intercept_
coef = modelo.coef_[0]

print(f"Modelo: tarifa = {intercept:.2f} + {coef:.3f} * edad")
print(f"Interpretación: cada año más de edad se asocia con £{coef:.2f} más de tarifa.")

# Predicción
edad_prueba = np.array([[25], [50]])
pred = modelo.predict(edad_prueba)
print(f"\nPredicción: edad=25 → tarifa={pred[0]:.2f}, edad=50 → tarifa={pred[1]:.2f}")

### 🟡 Ejercicio 2 — Aplicado · Regresión múltiple

⏱️ ~10 min

Entrena una regresión lineal múltiple para **predecir tarifa** usando **edad, clase y sexo** (codificar sexo como 0/1).

Pasos:
1. Crea features X con age, pclass, y sex (convertido a int).
2. Usa `train_test_split` con test_size=0.2, random_state=42.
3. Entrena LinearRegression().
4. Predice en test y reporta R².

In [ ]:
# 🟡 Regresión múltiple

df_model = titanic.copy()
df_model["sex_int"] = (df_model["sex"] == "female").astype(int)

X = df_model[["age", "pclass", "sex_int"]]
y = df_model["fare"]

# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar
lr = LinearRegression()
lr.___(X_tr, y_tr)  # ← completa

# Evaluar
pred = lr.___(X_te)  # ← completa
r2 = r2_score(y_te, pred)
mae = mean_absolute_error(y_te, pred)

print(f"R² en test = {r2:.3f}")
print(f"MAE = {mae:.2f} (error promedio en £)")
print(f"\nCoeficientes:")
for name, coef in zip(X.columns, lr.coef_):
    print(f"  {name}: {coef:+.3f}")
print(f"  intercepto: {lr.intercept_:+.3f}")

In [ ]:
assert r2 > 0, "R² debe ser positivo"
print("✅ Ejercicio 2 completado")

In [ ]:
# %% [solution]
# lr.fit(X_tr, y_tr)
# pred = lr.predict(X_te)

---

## 3. Regresión logística · clasificación

⏱️ ~10 min

> 📌 **Concepto clave:** Regresión logística predice probabilidades [0,1] usando la función sigmoidea. Ideal para clasificación binaria.

In [ ]:
# Demo: predecir supervivencia

df_clf = titanic.dropna(subset=["survived"]).copy()
df_clf["sex_int"] = (df_clf["sex"] == "female").astype(int)

X = df_clf[["age", "pclass", "sex_int"]]
y = df_clf["survived"]

# Train/test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar
clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)

# Evaluar
pred_class = clf.predict(X_te)
accuracy = accuracy_score(y_te, pred_class)

print(f"Accuracy en test = {accuracy:.3f}")
print(f"\nReporte de clasificación:")
print(classification_report(y_te, pred_class, target_names=["Murió", "Sobrevivió"]))

# Probabilidades
pred_proba = clf.predict_proba(X_te[:3])
print(f"\nPrimeras 3 probabilidades predichas:")
print(pd.DataFrame(pred_proba, columns=["P(murió)", "P(sobrevivió)"]))

### 🟡 Ejercicio 3 — Aplicado · Logística con features diferentes

⏱️ ~8 min

Entrena un LogisticRegression para predecir **supervivencia** usando **edad, clase, sexo y tarifa**.

Reporta accuracy y los 2 features más importantes (coeficientes en valor absoluto).

In [ ]:
# 🟡 Logística con fare

X = df_clf[["age", "pclass", "sex_int", "fare"]]
y = df_clf["survived"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

clf2 = LogisticRegression(max_iter=1000)
clf2.___(X_tr, y_tr)  # ← fit

acc = accuracy_score(y_te, clf2.___(X_te))  # ← predict
print(f"Accuracy = {acc:.3f}")

# Features importantes
coefs = pd.Series(abs(clf2.coef_[0]), index=X.columns).sort_values(ascending=False)
print(f"\nFeatures más importantes:")
print(coefs)

In [ ]:
assert acc > 0
print("✅ Ejercicio 3 completado")

In [ ]:
# %% [solution]
# clf2.fit(X_tr, y_tr)
# acc = accuracy_score(y_te, clf2.predict(X_te))

---

## 🔴 Ejercicio 4 — Desafío · Pipeline completo con estandarización

⏱️ ~15-20 min · Open-ended

Construye un pipeline completo:

1. **Preparación de datos:** limpieza, encoding categóricas, selección de features.
2. **Estandarización:** usa StandardScaler (importante para interpretabilidad).
3. **Entrenamiento:** elige regresión (predecir variable numérica) o clasificación (predecir categórica).
4. **Evaluación:** reporta métricas y top 3 features importantes.
5. **Interpretación:** 3-4 oraciones explicando qué aprendiste del modelo.

**Criterios:**
- ✓ Train/test split con random_state=42
- ✓ Estandarización antes de entrenar
- ✓ Métrica apropiada (R² o accuracy)
- ✓ Interpretación de coeficientes
- ✓ Conclusión en prosa

In [ ]:
# 🔴 Tu turno — Pipeline completo

# === PASO 1: Preparación ===
# Escoge regresión (objetivo numérico) o clasificación (objetivo categórico)
# Dataset recomendado: titanic, tips, diamonds, penguins

# ← Carga y prepara datos

# === PASO 2: Feature engineering ===
# Elige 4-6 features numéricas o codifica categóricas

# ← Crea X (features) e y (objetivo)

# === PASO 3: Train/test split ===
# X_tr, X_te, y_tr, y_te = ...

# ← Código aquí

# === PASO 4: Estandarización ===
# scaler = StandardScaler().fit(X_tr)
# X_tr_scaled = scaler.transform(X_tr)
# X_te_scaled = scaler.transform(X_te)

# ← Código aquí

# === PASO 5: Entrenamiento y evaluación ===
# Elige: LinearRegression o LogisticRegression

# ← Código aquí

# === PASO 6: Features importantes ===
# Reporta top 3 con coeficientes

# ← Código aquí

# === PASO 7: Conclusión ===
# En 3-4 oraciones: ¿qué aprendiste? ¿cuáles features importan? ¿es buen modelo?

# ← Escribe aquí

print("\n🎯 Ejercicio 4: Entregado (revisión manual recomendada)")

---

## 🧭 Checkpoint · Auto-evaluación

1. ¿Cuál es la diferencia entre Pearson y Spearman?
   - a) Pearson para lineal, Spearman para monotónica ← ✓
   - b) Son idénticos ← ✗

2. ¿Por qué estandarizar antes de entrenar?
   - a) Para que los coeficientes sean comparables ← ✓
   - b) No es necesario ← ✗

3. ¿Qué significa R²=0.75 en regresión?
   - a) El modelo explica 75% de la varianza en y ← ✓
   - b) El error es 75% ← ✗

📊 Has dominado regresión, correlación y scikit-learn. ¡Próxima: metricas de clasificación!

In [ ]:
assert "lr" in dir() and "clf" in dir(), "Completa los ejercicios"
print("Checkpoint ✓")

---

## 🧠 Mental model de hoy

```
Datos →  Exploración (correlación) → Pregunta (regresión o clasificación)
    ↓
  Feature engineering
    ↓
  Train/test split (random_state=42)
    ↓
  Estandarización (StandardScaler)
    ↓
  Fit modelo (LinearRegression | LogisticRegression)
    ↓
  Predict en test
    ↓
  Evaluar (R² | accuracy)
    ↓
  Interpretar coeficientes → Conclusiones
```

## 📌 Resumen

- **Pearson:** correlación lineal, Spearman para monotónica.
- **LinearRegression:** predice y numérico, coeficientes interpretables.
- **LogisticRegression:** predice probabilidades [0,1], ideal para clasificación.
- **Train/test split:** NUNCA evaluar en train (data leakage).
- **StandardScaler:** estandariza features para comparar coeficientes.
- **Scikit-learn:** API uniforme: fit, predict, score.

## 🚀 Próximos pasos

Mañana: **métricas de clasificación** (precision, recall, F1, confusion matrix) y comparación de modelos. Prepara: tener claras tus features para tu caso del proyecto.

## 📚 Para profundizar

- [Scikit-learn docs · LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [Scikit-learn docs · LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Hands-On ML with Scikit-Learn (Aurélien Géron)](https://github.com/ageron/handson-ml2) — cap 4: training linear models

## ✍️ Reflexión final

> Explica en 2-3 oraciones por qué separar train y test es más importante que elegir el mejor modelo.

---

## 🧑‍🤝‍🧑 Trabajo en grupo

- **Hito 1:** Matriz de correlación de vuestro dataset del proyecto.
- **Hito 2:** Identificar 3 features más prometedoras para predecir el objetivo.
- **Hito 3:** Entrenar regresión o clasificación baseline con esos features.
- **Entrega:** `baseline_model.ipynb` con datos, modelo, métrica y top 3 features.